# Notebook 1 - Data Pipeline
Owner: Reema Eid - Data Engineer

---

### What this notebook does

This notebook pulls all the raw data the PaySprint agent needs:
- Stock prices and company fundamentals from Yahoo Finance
- Financial news from GNews
- Sentiment scores on each article
- Technical indicators (momentum, volatility)

How to use this notebook:
1. Run every cell from top to bottom (Shift+Enter on each cell, or Run All)
2. Look for the sections marked YOUR TURN - those are yours to customize
3. You do not need to change anything else - everything already works

---

### First time only - install dependencies

Run the cell below once, then skip it in future runs.

In [ ]:
import subprocess, sys
packages = [
    'yfinance', 'gnews', 'nltk', 'pandas', 'numpy',
    'scikit-learn', 'python-dotenv', 'openai'
]
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import nltk
nltk.download('vader_lexicon', quiet=True)
print('All packages installed.')

In [ ]:
import sys, os
sys.path.insert(0, '.')

from paysprint_agent import (
    fetch_price_history,
    fetch_fundamentals,
    fetch_news,
    compute_indicators,
    score_sentiment,
    init_db,
    save_user_profile,
    TRUSTED_SOURCES,
    SCREENER_STOCKS,
)
import paysprint_agent as core
import pandas as pd
print('Core module loaded.')

---
## Step 1 - Define the Test Investor Profile

This profile is used for testing throughout the notebook.

In [ ]:
SAMPLE_PROFILE = {
    'name':              'Test Investor',
    'budget':            5000.0,
    'aggressiveness':    'moderate',   # conservative | moderate | aggressive
    'horizon_months':    12,
    'current_holdings':  {},
    'preferred_sectors': ['Technology'],
}

print('Sample profile:')
for k, v in SAMPLE_PROFILE.items():
    print(f'  {k}: {v}')

---
## Step 2 - Stock Price Data

We pull 90 days of closing prices from Yahoo Finance.
No API key needed - Yahoo Finance is free.

In [ ]:
TEST_TICKERS = ['AAPL', 'MSFT', 'NVDA', 'JNJ', 'GOOGL']

print('Fetching 90-day price history...\n')
price_data = {}
for ticker in TEST_TICKERS:
    prices = fetch_price_history(ticker, lookback_days=90)
    price_data[ticker] = prices
    if not prices.empty:
        print(f'  {ticker}: ${prices.iloc[-1]:.2f} (latest close), {len(prices)} trading days')
    else:
        print(f'  {ticker}: No data returned')

In [ ]:
# View the AAPL price history as a table
aapl_prices = price_data['AAPL']
print('AAPL - last 10 trading days:')
print(aapl_prices.tail(10).to_string())

---
## Step 3 - Company Fundamentals

We pull key financial metrics: P/E ratio, EPS, revenue growth, debt, market cap, sector.

In [ ]:
print('Fetching fundamentals...\n')
fund_records = []
for ticker in TEST_TICKERS:
    f = fetch_fundamentals(ticker)
    fund_records.append(f)

fund_df = pd.DataFrame(fund_records)
print(fund_df[['ticker', 'sector', 'market_cap', 'pe_ratio', 'revenue_growth', 'eps']].to_string(index=False))

---
## Step 4 - Financial News and Sentiment

We fetch recent financial news using GNews and score each article's sentiment
using VADER (a pre-trained sentiment model - no training needed).

Sentiment score: -1.0 = very negative, 0.0 = neutral, +1.0 = very positive

In [ ]:
print('Fetching news for NVDA (last 14 days)...\n')
news_df = fetch_news(query='NVDA NVIDIA', days=14, max_results=20)

if news_df.empty:
    print('No news returned. GNews may be rate-limiting. Try again in a few minutes.')
else:
    print(f'Found {len(news_df)} articles\n')
    display_cols = ['title', 'publisher', 'trusted', 'sentiment']
    print(news_df[display_cols].head(10).to_string(index=False))

In [ ]:
sample_headlines = [
    'NVIDIA beats earnings expectations by 40 percent - record quarter',
    'Stock market crashes on recession fears',
    'Apple announces new product launch next quarter',
]

print('Sentiment scoring examples:\n')
for headline in sample_headlines:
    score = score_sentiment(headline)
    label = 'POSITIVE' if score > 0.1 else ('NEGATIVE' if score < -0.1 else 'NEUTRAL')
    print(f'  [{label:8s}] {score:+.2f}  "{headline}"')

---
## Step 5 - Technical Indicators

We compute momentum and volatility for each stock.
The slope tells us if prices are trending up (positive) or down (negative).

In [ ]:
print('Computing technical indicators...\n')
indicator_records = []
for ticker in TEST_TICKERS:
    ind = compute_indicators(ticker, lookback_days=120)
    indicator_records.append(ind)

ind_df = pd.DataFrame(indicator_records)
cols   = ['ticker', 'last_price', 'slope_per_day', 'forecast_3m', 'forecast_12m', 'volatility_30d']
cols   = [c for c in cols if c in ind_df.columns]
print(ind_df[cols].to_string(index=False))

---
## Step 6 - Save to Database

We save the test profile to a local SQLite database.

In [ ]:
import sqlite3

init_db()
user_id = save_user_profile(SAMPLE_PROFILE)
print(f'Profile saved to paysprint.db  (user_id = {user_id})')

conn = sqlite3.connect('paysprint.db')
df   = pd.read_sql('SELECT * FROM user_profiles', conn)
conn.close()
print(df.to_string(index=False))

---
## YOUR TURN - Reema's Customizations

Everything above already works. The tasks below are yours to explore and improve.
Pick any task - each one is independent.

---
### Task A - Add more tickers to the stock screener

The screener has 8 stocks per strategy tier. Add more tickers to any tier.

Good sources to find tickers: Yahoo Finance, Google Finance, or search the company name + stock ticker.

Recommendation: Add 2-3 tickers per tier that are not already in the list.

In [ ]:
# Show current screener tickers
print('Current screener stocks:')
for strategy, tickers in core.SCREENER_STOCKS.items():
    print(f'  {strategy}: {tickers}')
print()

# YOUR TURN: Add your tickers to the lists below
# Examples of what you could add:
#   conservative: 'COST' (Costco), 'BRK-B' (Berkshire Hathaway)
#   moderate:     'AMGN' (Amgen), 'NFLX' (Netflix)
#   aggressive:   'SOFI' (SoFi Technologies), 'RKLB' (Rocket Lab)

core.SCREENER_STOCKS['conservative'].extend(['COST', 'BRK-B'])
core.SCREENER_STOCKS['moderate'].extend(['AMGN', 'NFLX'])
core.SCREENER_STOCKS['aggressive'].extend(['SOFI', 'RKLB'])

print('Updated screener stocks:')
for strategy, tickers in core.SCREENER_STOCKS.items():
    print(f'  {strategy}: {tickers}')

In [ ]:
# Verify your new tickers have price data
all_tickers = [t for tickers in core.SCREENER_STOCKS.values() for t in tickers]

print('Checking price data for all screener tickers...\n')
for ticker in sorted(set(all_tickers)):
    prices = fetch_price_history(ticker, lookback_days=5)
    status = f'${prices.iloc[-1]:.2f}' if not prices.empty else 'No data - consider removing this ticker'
    print(f'  {ticker:8s} {status}')

---
### Task B - Add more trusted news sources

The agent filters out news from unknown publishers.
Add more sources to expand what the agent considers reliable.

Recommendation: Add 3-5 more financial news sites you know and trust.

In [ ]:
print(f'Current trusted sources ({len(core.TRUSTED_SOURCES)} total):')
print(sorted(core.TRUSTED_SOURCES))

In [ ]:
# YOUR TURN: Add your trusted sources here
# Use lowercase. Partial names work (e.g. 'techcrunch' matches 'TechCrunch').
# Tip: run the news fetch in Step 4 and look at the publisher column to find names.

new_sources = [
    'the information',
    'financial post',
    # add more here...
]

core.TRUSTED_SOURCES.update(new_sources)
print(f'Added {len(new_sources)} sources. Total: {len(core.TRUSTED_SOURCES)}')

---
### Task C - Explore news for different stocks

Try fetching news for different tickers and compare sentiment scores.

Recommendation: Try at least 2 different stocks from the screener.

In [ ]:
# YOUR TURN: Change the tickers in this list to any stocks you want to explore
explore_tickers = ['AAPL', 'TSLA', 'JNJ']

print('News sentiment comparison:\n')
sentiment_summary = []
for ticker in explore_tickers:
    news = fetch_news(query=ticker, days=14, max_results=15)
    if news.empty:
        avg_sent = 0.0
        count    = 0
    else:
        avg_sent = news['sentiment'].mean()
        count    = len(news)
    label = 'POSITIVE' if avg_sent > 0.05 else ('NEGATIVE' if avg_sent < -0.05 else 'NEUTRAL')
    sentiment_summary.append({'ticker': ticker, 'articles': count, 'avg_sentiment': round(avg_sent, 3), 'label': label})
    print(f'  {ticker:6s}: {count:3d} articles | avg sentiment = {avg_sent:+.3f} [{label}]')

sentiment_df = pd.DataFrame(sentiment_summary)
print('\nSummary table:')
print(sentiment_df.to_string(index=False))

---
### Task D - Write your data quality observations

Reema: Replace the placeholder text below with your own observations.

After running the notebook, write 3-5 sentences answering:
- Which stocks had the most news coverage?
- Did any tickers have missing or unexpected data?
- What patterns did you notice in the sentiment scores?
- Are there any data sources you think would improve the pipeline?

This commentary will be used in the video presentation.

**Reema's Data Quality Commentary**

Write your observations here after running the notebook.

Example: NVDA had the highest news coverage with 18 articles over 14 days, all from trusted sources.
JNJ had very few articles, which may reduce the reliability of its sentiment score.
Some smaller tickers returned no price data and should be removed from the screener.

In [ ]:
# Final summary - export all data to CSV for Notebook 2
import os
os.makedirs('data', exist_ok=True)

print('Building master data table...\n')
master_records = []
for ticker in TEST_TICKERS:
    f    = fetch_fundamentals(ticker)
    ind  = compute_indicators(ticker)
    news = fetch_news(query=ticker, days=14, max_results=15)
    avg_sent = round(news['sentiment'].mean(), 3) if not news.empty else 0.0
    row = {
        'ticker':          ticker,
        'sector':          f.get('sector'),
        'market_cap':      f.get('market_cap'),
        'pe_ratio':        f.get('pe_ratio'),
        'revenue_growth':  f.get('revenue_growth'),
        'last_price':      ind.get('last_price'),
        'slope_per_day':   ind.get('slope_per_day'),
        'volatility_30d':  ind.get('volatility_30d'),
        'avg_sentiment':   avg_sent,
    }
    master_records.append(row)

master_df = pd.DataFrame(master_records)
master_df.to_csv('data/master_stock_data.csv', index=False)
print('Saved to data/master_stock_data.csv')
print(master_df.to_string(index=False))